In [ ]:
# Import required libraries
import glob
import geopandas as gpd
import os
import osmnx as ox
import time

# Configure osmnx for newer versions
ox.settings.use_cache = True
ox.settings.log_console = True

# Load AOI boundaries
print("Loading AOI boundaries...")
aoi_files = glob.glob(os.path.join('..', 'outputs', 'City_*', '*_AOI_Boundary.geojson'))
aoi_dict = {}

for file in aoi_files:
    city_name = os.path.basename(os.path.dirname(file)).replace('City_', '')
    gdf = gpd.read_file(file)
    aoi_dict[city_name] = gdf
    print(f"✓ Loaded AOI for {city_name}")

# Process each city sequentially
cities_to_process = sorted(aoi_dict.keys())

for i, city in enumerate(cities_to_process, 1):
    print(f"\n{'='*70}")
    print(f"PROCESSING {city.upper()} ({i}/{len(cities_to_process)})")
    print(f"{'='*70}")
    
    gdf = aoi_dict[city]
    bounds = gdf.total_bounds  # (minx, miny, maxx, maxy)
    
    # Create output directory
    city_data_dir = os.path.join('..', 'data', f'City_{city}')
    os.makedirs(city_data_dir, exist_ok=True)
    
    # 1. DOWNLOAD MAJOR ROADS AND EXPRESSWAYS
    try:
        print(f"\n1. Downloading major roads for {city}...")
        
        # Define major road types in OSM
        major_roads_tags = {
            'highway': [
                'motorway', 'motorway_link',      # Expressways
                'trunk', 'trunk_link',            # Major highways
                'primary', 'primary_link',        # Primary roads
                'secondary', 'secondary_link',    # Secondary roads
                'tertiary', 'tertiary_link'       # Tertiary roads
            ]
        }
        
        # Download major roads
        major_roads = ox.features_from_bbox(
            bbox=(bounds[1], bounds[0], bounds[3], bounds[2]),
            tags=major_roads_tags
        )
        
        if not major_roads.empty:
            # Save all major roads
            roads_file = os.path.join(city_data_dir, f'{city}_major_roads.geojson')
            major_roads.to_file(roads_file, driver='GeoJSON')
            print(f"   ✓ Saved {len(major_roads)} major road segments")
            
            # Show road type breakdown
            if 'highway' in major_roads.columns:
                road_types = major_roads['highway'].value_counts()
                print("   Road types found:")
                for road_type, count in road_types.items():
                    print(f"     - {road_type}: {count} segments")
        else:
            print(f"   - No major roads found")
            
    except Exception as e:
        print(f"   ✗ Error downloading major roads: {e}")
    
    # 2. DOWNLOAD BUILT-UP AREAS
    try:
        print(f"\n2. Downloading built-up areas for {city}...")
        
        # Built-up areas in OSM are typically landuse=residential/commercial/industrial
        built_up_tags = {
            'landuse': [
                'residential',
                'commercial', 
                'industrial',
                'retail',
                'construction'
            ]
        }
        
        built_up_areas = ox.features_from_bbox(
            bbox=(bounds[1], bounds[0], bounds[3], bounds[2]),
            tags=built_up_tags
        )
        
        if not built_up_areas.empty:
            # Save built-up areas
            built_up_file = os.path.join(city_data_dir, f'{city}_built_up_areas.geojson')
            built_up_areas.to_file(built_up_file, driver='GeoJSON')
            print(f"   ✓ Saved {len(built_up_areas)} built-up area polygons")
            
            # Show landuse breakdown
            if 'landuse' in built_up_areas.columns:
                landuse_types = built_up_areas['landuse'].value_counts()
                print("   Built-up area types:")
                for landuse_type, count in landuse_types.items():
                    print(f"     - {landuse_type}: {count} areas")
        else:
            print(f"   - No built-up areas found")
            
    except Exception as e:
        print(f"   ✗ Error downloading built-up areas: {e}")
    
    # 3. DOWNLOAD PLACES/VILLAGES
    try:
        print(f"\n3. Downloading places and villages for {city}...")
        
        # Places in OSM are tagged with place=*
        place_tags = {
            'place': [
                'city',
                'town',
                'village',
                'hamlet',
                'suburb',
                'neighbourhood',
                'locality'
            ]
        }
        
        places = ox.features_from_bbox(
            bbox=(bounds[1], bounds[0], bounds[3], bounds[2]),
            tags=place_tags
        )
        
        if not places.empty:
            # Save places
            places_file = os.path.join(city_data_dir, f'{city}_places_villages.geojson')
            places.to_file(places_file, driver='GeoJSON')
            print(f"   ✓ Saved {len(places)} places/villages")
            
            # Show place types breakdown
            if 'place' in places.columns:
                place_types = places['place'].value_counts()
                print("   Place types found:")
                for place_type, count in place_types.items():
                    print(f"     - {place_type}: {count} locations")
            
            # Show some place names if available
            if 'name' in places.columns:
                named_places = places[places['name'].notna()]['name'].head(10)
                if not named_places.empty:
                    print("   Sample place names:")
                    for name in named_places:
                        print(f"     - {name}")
        else:
            print(f"   - No places/villages found")
            
    except Exception as e:
        print(f"   ✗ Error downloading places: {e}")
    
    # Small delay to ensure clean output
    time.sleep(0.5)

# Print final summary
print(f"\n{'='*70}")
print("DOWNLOAD COMPLETE - SUMMARY")
print(f"{'='*70}")

for city in cities_to_process:
    city_data_dir = os.path.join('..', 'data', f'City_{city}')
    
    if os.path.exists(city_data_dir):
        files = os.listdir(city_data_dir)
        geojson_files = [f for f in files if f.endswith('.geojson')]
        
        print(f"\n{city}:")
        total_size = 0
        for file in sorted(geojson_files):
            file_path = os.path.join(city_data_dir, file)
            file_size = os.path.getsize(file_path) / 1024 / 1024  # MB
            total_size += file_size
            print(f"  - {file} ({file_size:.2f} MB)")
        print(f"  Total: {total_size:.2f} MB")
    else:
        print(f"\n{city}: No data downloaded")

print(f"\n{'='*70}")

In [ ]:
# First, make sure we have Lagos data loaded
lagos_gdf = aoi_dict['Lagos']
city_data_dir = os.path.join('..', 'data', 'City_Lagos')
os.makedirs(city_data_dir, exist_ok=True)

print("Lagos AOI loaded successfully")
print(f"Lagos bounds: {lagos_gdf.total_bounds}")
print(f"Lagos CRS: {lagos_gdf.crs}")

# Ensure Lagos is in WGS84
if str(lagos_gdf.crs) != "EPSG:4326":
    print("Converting Lagos to WGS84...")
    lagos_gdf = lagos_gdf.to_crs("EPSG:4326")

# Method 1: Try downloading using place query
print("\nDownloading Lagos data using place query...")

# 1. Roads
try:
    print("1. Downloading Lagos road network...")
    G = ox.graph_from_place("Lagos, Nigeria", network_type='drive')
    
    # Convert to GeoDataFrame
    edges = ox.graph_to_gdfs(G, nodes=False, edges=True)
    
    # Clip to AOI
    edges_clipped = gpd.clip(edges, lagos_gdf)
    
    # Filter major roads
    if 'highway' in edges_clipped.columns:
        major_types = ['motorway', 'trunk', 'primary', 'secondary', 'tertiary',
                      'motorway_link', 'trunk_link', 'primary_link', 'secondary_link', 'tertiary_link']
        major_roads = edges_clipped[edges_clipped['highway'].isin(major_types)]
        
        if not major_roads.empty:
            roads_file = os.path.join(city_data_dir, 'Lagos_major_roads.geojson')
            major_roads.to_file(roads_file, driver='GeoJSON')
            print(f"   ✓ Saved {len(major_roads)} major road segments")
            
            # Show breakdown
            road_types = major_roads['highway'].value_counts()
            print("   Road types found:")
            for road_type, count in road_types.head(10).items():
                print(f"     - {road_type}: {count} segments")
        else:
            print("   - No major roads found after filtering")
    else:
        print("   - No highway column found in data")

except Exception as e:
    print(f"   ✗ Error downloading roads: {e}")

# 2. Built-up areas
try:
    print("\n2. Downloading Lagos built-up areas...")
    built_up = ox.features_from_place(
        "Lagos, Nigeria",
        tags={'landuse': ['residential', 'commercial', 'industrial', 'retail', 'construction']}
    )
    
    if not built_up.empty:
        # Clip to AOI
        built_up_clipped = gpd.clip(built_up, lagos_gdf)
        
        if not built_up_clipped.empty:
            built_up_file = os.path.join(city_data_dir, 'Lagos_built_up_areas.geojson')
            built_up_clipped.to_file(built_up_file, driver='GeoJSON')
            print(f"   ✓ Saved {len(built_up_clipped)} built-up area polygons")
            
            # Show breakdown
            if 'landuse' in built_up_clipped.columns:
                landuse_types = built_up_clipped['landuse'].value_counts()
                print("   Built-up area types:")
                for landuse_type, count in landuse_types.items():
                    print(f"     - {landuse_type}: {count} areas")
        else:
            print("   - No built-up areas within AOI")
    else:
        print("   - No built-up areas found")

except Exception as e:
    print(f"   ✗ Error downloading built-up areas: {e}")

# 3. Places
try:
    print("\n3. Downloading Lagos places and villages...")
    places = ox.features_from_place(
        "Lagos, Nigeria",
        tags={'place': ['city', 'town', 'village', 'hamlet', 'suburb', 'neighbourhood', 'locality']}
    )
    
    if not places.empty:
        # Clip to AOI
        places_clipped = gpd.clip(places, lagos_gdf)
        
        if not places_clipped.empty:
            places_file = os.path.join(city_data_dir, 'Lagos_places_villages.geojson')
            places_clipped.to_file(places_file, driver='GeoJSON')
            print(f"   ✓ Saved {len(places_clipped)} places/villages")
            
            # Show breakdown
            if 'place' in places_clipped.columns:
                place_types = places_clipped['place'].value_counts()
                print("   Place types found:")
                for place_type, count in place_types.items():
                    print(f"     - {place_type}: {count} locations")
            
            # Show sample names
            if 'name' in places_clipped.columns:
                named_places = places_clipped[places_clipped['name'].notna()]['name'].head(10)
                if not named_places.empty:
                    print("   Sample place names:")
                    for name in named_places:
                        print(f"     - {name}")
        else:
            print("   - No places within AOI")
    else:
        print("   - No places found")

except Exception as e:
    print(f"   ✗ Error downloading places: {e}")

# Print summary for Lagos
print(f"\n{'='*70}")
print("LAGOS DOWNLOAD SUMMARY")
print(f"{'='*70}")

if os.path.exists(city_data_dir):
    files = os.listdir(city_data_dir)
    geojson_files = [f for f in files if f.endswith('.geojson')]
    
    total_size = 0
    for file in sorted(geojson_files):
        file_path = os.path.join(city_data_dir, file)
        file_size = os.path.getsize(file_path) / 1024 / 1024  # MB
        total_size += file_size
        print(f"  - {file} ({file_size:.2f} MB)")
    print(f"\nTotal: {total_size:.2f} MB")
else:
    print("No data directory found")

print(f"{'='*70}")